In [3]:
import sqlite3
import os
import json
import urllib.parse
from rdflib import Graph, Literal, RDF, URIRef, Namespace, Literal, RDF, RDFS, XSD, OWL
from rdflib.namespace import RDFS, XSD, SKOS

# Define Schema

In [98]:
g = Graph()

# Standard Vocabularies
OBAN = Namespace("http://purl.org/oban/")
UP = Namespace("http://purl.uniprot.org/core/")

# YOUR Custom Project Namespace
MTR = Namespace("https://metabolite-ratio-app.unil.ch/")
g.bind("ex", MTR)
g.bind("oban", OBAN)
g.bind("up", UP)
g.bind("owl", OWL)

# 2. Define the Ontology Metadata
ontology_uri = URIRef("https://metabolite-ratio-app.unil.ch/ontology")
g.add((ontology_uri, RDF.type, OWL.Ontology))
g.add((ontology_uri, RDFS.label, Literal("rQTL Metabolic Ratio Ontology")))
g.add((ontology_uri, RDFS.comment, Literal("A schema defining genetically regulated metabolic reactions.")))

# ==========================================
# 3. DEFINE CLASSES (The "Nouns")
# ==========================================
classes = {
    "MetaboliteRatio": "A ratio between two metabolite concentrations.",
    "Metabolite": "A small chemical compound.",
    "SNP": "Single Nucleotide Polymorphism.",
    "Gene": "A region of DNA.",
    "Phenotype": "An observable clinical trait or disease."
}

for cls_name, description in classes.items():
    node = MTR[cls_name]
    g.add((node, RDF.type, OWL.Class))
    g.add((node, RDFS.label, Literal(cls_name)))
    g.add((node, RDFS.comment, Literal(description)))

# Integrate UniProt Enzyme class (as discussed previously)
g.add((MTR.Gene, RDFS.subClassOf, UP.Protein)) # A gene maps to a protein/enzyme

# ==========================================
# 4. DEFINE OBJECT PROPERTIES (The "Edges" between nodes)
# Here we define the Domain (starts at) and Range (ends at)
# ==========================================

# Ratio -> Metabolite
g.add((MTR.has_numerator, RDF.type, OWL.ObjectProperty))
g.add((MTR.has_numerator, RDFS.domain, MTR.MetaboliteRatio))
g.add((MTR.has_numerator, RDFS.range, MTR.Metabolite))

g.add((MTR.has_denominator, RDF.type, OWL.ObjectProperty))
g.add((MTR.has_denominator, RDFS.domain, MTR.MetaboliteRatio))
g.add((MTR.has_denominator, RDFS.range, MTR.Metabolite))

# SNP -> Gene
g.add((MTR.closest_gene, RDF.type, OWL.ObjectProperty))
g.add((MTR.closest_gene, RDFS.domain, MTR.SNP))
g.add((MTR.closest_gene, RDFS.range, MTR.Gene))

g.add((MTR.is_eQTL_for, RDF.type, OWL.ObjectProperty))
g.add((MTR.is_eQTL_for, RDFS.domain, MTR.SNP))
g.add((MTR.is_eQTL_for, RDFS.range, MTR.Gene))

# SNP -> Phenotype
g.add((MTR.associated_with_trait, RDF.type, OWL.ObjectProperty))
g.add((MTR.associated_with_trait, RDFS.domain, MTR.SNP))
g.add((MTR.associated_with_trait, RDFS.range, MTR.Phenotype))

# ==========================================
# 5. DEFINE DATATYPE PROPERTIES (The "Edges" to values/numbers)
# ==========================================

# Association -> Float
g.add((MTR.p_value, RDF.type, OWL.DatatypeProperty))
g.add((MTR.p_value, RDFS.domain, OBAN.association))
g.add((MTR.p_value, RDFS.range, XSD.float))

g.add((MTR.beta, RDF.type, OWL.DatatypeProperty))
g.add((MTR.beta, RDFS.domain, OBAN.association))
g.add((MTR.beta, RDFS.range, XSD.float))

g.add((MTR.pGain, RDF.type, OWL.DatatypeProperty))
g.add((MTR.pGain, RDFS.domain, OBAN.association))
g.add((MTR.pGain, RDFS.range, XSD.float))


<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>

In [99]:
# Add standard namespaces
BIOLINK = Namespace("https://w3id.org/biolink/vocab/")
SIO = Namespace("http://semanticscience.org/resource/")
SO = Namespace("http://purl.obolibrary.org/obo/SO_")
PROV = Namespace("http://www.w3.org/ns/prov#")

g.bind("biolink", BIOLINK)
g.bind("sio", SIO)
g.bind("so", SO)
g.bind("prov", PROV)

# 1. Map SNP to Sequence Ontology
g.add((MTR.SNP, RDFS.subClassOf, SO["0000694"])) # SO:0000694 is 'SNP'

# 2. Hybrid Ontology Approach: Map custom edges to Biolink
g.add((MTR.is_eQTL_for, RDFS.subPropertyOf, BIOLINK.affects_expression_of))
g.add((MTR.associated_with_trait, RDFS.subPropertyOf, BIOLINK.associated_with))

# 3. Hybrid Ontology Approach: Map statistics to SIO
g.add((MTR.p_value, RDFS.subPropertyOf, SIO["000765"])) # SIO:000765 is 'p-value'
g.add((MTR.beta, RDFS.subPropertyOf, SIO["001078"])) # SIO:001078 is 'beta' (or similar estimate)

# 4. Add PROV-O metadata to your generated dataset
g.add((MTR.Dataset, RDF.type, PROV.Entity))
g.add((MTR.Dataset, PROV.wasGeneratedBy, Literal("Python JSON-to-RDF parser")))

<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>

In [100]:
sql_database_path = './data/metabolite-ratio-app.sqlite'

In [101]:
if not os.path.exists(sql_database_path):
    print("not")

not


In [102]:
import json
import urllib.parse
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import RDFS, XSD, SKOS

# 1. Define standard vocabularies and YOUR custom schema
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
HMDB = Namespace("http://identifiers.org/hmdb/")
DBSNP = Namespace("http://identifiers.org/dbsnp/")
HGNC = Namespace("http://identifiers.org/hgnc.symbol/")
OBAN = Namespace("http://purl.org/oban/")

# Your custom ontology namespace
MTR = Namespace("http://example.org/rqtl/")

def clean_uri_string(text):
    """Utility to clean strings so they can be valid URIs."""
    return urllib.parse.quote(str(text).strip().replace(" ", "_").replace('"', ''))

def add_rqtl_to_graph(json_file_path, g):
    # 2. Bind Namespaces
    g.bind("chebi", CHEBI)
    g.bind("hmdb", HMDB)
    g.bind("dbsnp", DBSNP)
    g.bind("hgnc", HGNC)
    g.bind("oban", OBAN)
    g.bind("mtr", MTR)
    g.bind("skos", SKOS)

    with open(json_file_path, 'r') as f:
        data = json.load(f)

    for entry in data:
        ratio_id = entry['ratio_accession']
        ratio_node = MTR[ratio_id]

        # Define Ratio Node
        g.add((ratio_node, RDF.type, MTR.MetaboliteRatio))
        g.add((ratio_node, RDFS.label, Literal(entry['ratio_name'])))

        # --- Add Numerator & Denominator ---
        for side, relation in [('numerator_metabolite', MTR.has_numerator),
                               ('denominator_metabolite', MTR.has_denominator)]:
            metab = entry.get(side, {})
            if metab and metab.get('chebi') != "NA":
                chebi_id = str(int(metab['chebi']))
                metab_node = CHEBI[chebi_id]
                g.add((metab_node, RDF.type, MTR.Metabolite))
                g.add((metab_node, RDFS.label, Literal(metab['name'])))
                if metab.get('hmdb') != "NA":
                    g.add((metab_node, SKOS.exactMatch, HMDB[metab['hmdb']]))
                g.add((ratio_node, relation, metab_node))

        # --- Process SNPs and Statistics ---
        regions = entry.get('associated_regions', {})
        for rsid, details in regions.items():
            snp_node = DBSNP[rsid]
            g.add((snp_node, RDF.type, MTR.SNP))

            # OBAN Association Node (SNP -> Ratio)
            assoc_node = MTR[f"Assoc_{rsid}_{ratio_id}"]
            g.add((assoc_node, RDF.type, OBAN.association))
            g.add((assoc_node, OBAN.has_subject, snp_node))
            g.add((assoc_node, OBAN.has_object, ratio_node))

            stats = details.get('summary_statistics', {})
            if 'p_value' in stats:
                g.add((assoc_node, MTR.p_value, Literal(stats['p_value'], datatype=XSD.float)))
            if 'beta' in stats:
                g.add((assoc_node, MTR.beta, Literal(stats['beta'], datatype=XSD.float)))

            # =========================================================
            # NEW: PROCESS eQTL AND pQTL CAUSAL EVIDENCE (MR / COLOC)
            # =========================================================
            for qtl_type in ['eQTL', 'pQTL']:
                qtl_data = details.get(qtl_type, {})
                if isinstance(qtl_data, dict) and 'ratio' in qtl_data:

                    for causal_test in qtl_data['ratio']:
                        exposure_name = causal_test.get('exposure')
                        if not exposure_name or exposure_name == "NA":
                            continue

                        # Define the Exposure Node (Gene for eQTL, Protein for pQTL)
                        if qtl_type == 'eQTL':
                            exposure_node = HGNC[clean_uri_string(exposure_name)]
                            g.add((exposure_node, RDF.type, MTR.Gene))
                        else:
                            exposure_node = MTR[f"Protein_{clean_uri_string(exposure_name)}"]
                            g.add((exposure_node, RDF.type, MTR.Protein))

                        g.add((exposure_node, RDFS.label, Literal(exposure_name)))

                        # Create a Causal Assessment Node linking Exposure -> Ratio
                        assessment_id = f"CausalMR_{qtl_type}_{clean_uri_string(exposure_name)}_{ratio_id}"
                        assessment_node = MTR[assessment_id]

                        g.add((assessment_node, RDF.type, MTR.CausalAssessment))
                        g.add((assessment_node, MTR.assessment_type, Literal(qtl_type)))
                        g.add((assessment_node, MTR.has_exposure, exposure_node))
                        g.add((assessment_node, MTR.has_outcome, ratio_node))

                        # Attach Mendelian Randomization (MR) Statistics
                        if causal_test.get('beta_ivw') != "NA":
                            g.add((assessment_node, MTR.beta_ivw, Literal(causal_test['beta_ivw'], datatype=XSD.float)))
                        if causal_test.get('p_ivw') != "NA":
                            g.add((assessment_node, MTR.p_ivw, Literal(causal_test['p_ivw'], datatype=XSD.float)))

                        # Attach Colocalization Statistics (Posterior Probability H4)
                        if causal_test.get('PP.H4.abf') != "NA":
                            g.add((assessment_node, MTR.colocalization_pp4, Literal(causal_test['PP.H4.abf'], datatype=XSD.float)))

                        # Optional: Add tissue context if it exists (like in GTEx data)
                        if causal_test.get('tissue') and causal_test.get('tissue') != "NA":
                            g.add((assessment_node, MTR.measured_in_tissue, Literal(causal_test['tissue'])))
            # =========================================================

    return g



In [103]:
import glob
json_files= glob.glob("../data/json_files/*.json")
rqtl_json_path = json_files[0]
rqtl_json_path = "../data/json_files/GCST90200330_GCST90200020.json"

In [104]:
rqtl_json_path

'../data/json_files/GCST90200330_GCST90200020.json'

In [105]:
add_rqtl_to_graph(rqtl_json_path,g)

<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>

In [106]:
print(f"Graph created with {len(g)} triples.")


Graph created with 233 triples.


In [107]:
print(g.serialize(format="ttl"))

@prefix biolink: <https://w3id.org/biolink/vocab/> .
@prefix chebi: <http://purl.obolibrary.org/obo/CHEBI_> .
@prefix dbsnp: <http://identifiers.org/dbsnp/> .
@prefix ex: <https://metabolite-ratio-app.unil.ch/> .
@prefix hgnc: <http://identifiers.org/hgnc.symbol/> .
@prefix hmdb: <http://identifiers.org/hmdb/> .
@prefix mtr: <http://example.org/rqtl/> .
@prefix oban: <http://purl.org/oban/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sio: <http://semanticscience.org/resource/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix so: <http://purl.obolibrary.org/obo/SO_> .
@prefix up: <http://purl.uniprot.org/core/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

mtr:Assoc_rs102275_GCST90200330_GCST90200020 a oban:association ;
    mtr:beta "0.3355685"^^xsd:float ;
    oban:has_object mtr:GCST90200330_GCST90200020 ;
    oban:has_subject dbsnp:rs102275 .

mt

In [108]:
if __name__ == "__main__":
    # Ensure your JSON file is named correctly in the same directory
    json_file = "GCST90199675_GCST90200402.json"

    print("Building Knowledge Graph...")
    kg = build_rqtl_graph(json_file)

    print(f"Graph created with {len(kg)} triples.")

    # Save to Turtle format (.ttl)
    output_file = "rqtl_knowledge_graph.ttl"
    kg.serialize(destination=output_file, format="turtle")
    print(f"Graph saved successfully to {output_file}")

Building Knowledge Graph...


NameError: name 'build_rqtl_graph' is not defined

In [109]:
conn = sqlite3.connect('../data/metabolite-ratio-app.sqlite')
cursor = conn.cursor()

In [110]:
c = 100

rows = cursor.execute(f"SELECT * FROM ratios LIMIT {c}").fetchall()

In [111]:
rows[88]

('GCST90200168_GCST90200488',
 'hydroxyasparagine** to x-12117',
 'GCST90200168',
 'GCST90200488',
 '"NA"',
 1.148964929798397e+36,
 1,
 0,
 '117',
 1.0,
 'AGXT2 (1.000)',
 'N',
 'N',
 None,
 None,
 None,
 None,
 'NA')

# GWAS Catalog Integration

In [112]:
import requests
import time
import urllib.parse
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import RDFS, XSD

# --- Namespaces ---
DBSNP = Namespace("http://identifiers.org/dbsnp/")
EFO = Namespace("http://www.ebi.ac.uk/efo/")
BIOLINK = Namespace("https://w3id.org/biolink/vocab/")
PROV = Namespace("http://www.w3.org/ns/prov#")
SIO = Namespace("http://semanticscience.org/resource/")
MTR = Namespace("http://example.org/rqtl/") # Your custom namespace

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

def gwas_api_request(endpoint, params=None):
    """
    Performs a GET request to the GWAS Catalog REST API.
    """
    full_url = f"{BASE_URL}{endpoint}"
    try:
        response = requests.get(full_url, params=params, headers={"Accept": "application/json"})
        if response.status_code == 200:
            return response.json()
        else:
            return None
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None

import requests
import time
import urllib.parse
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import RDFS, XSD, SKOS

# --- Namespaces ---
DBSNP = Namespace("http://identifiers.org/dbsnp/")
EFO = Namespace("http://www.ebi.ac.uk/efo/")
BIOLINK = Namespace("https://w3id.org/biolink/vocab/")
PROV = Namespace("http://www.w3.org/ns/prov#")
SIO = Namespace("http://semanticscience.org/resource/")
PUBMED = Namespace("https://pubmed.ncbi.nlm.nih.gov/") # New namespace for literature
MTR = Namespace("http://example.org/rqtl/")

BASE_URL = "https://www.ebi.ac.uk/gwas/rest/api"

def gwas_api_request(endpoint, params=None):
    full_url = f"{BASE_URL.rstrip('/')}/{endpoint.lstrip('/')}"
    try:
        response = requests.get(full_url, params=params, headers={"Accept": "application/json"})
        if response.status_code == 200:
            return response.json()
        # Handle 404s gracefully without crashing
        elif response.status_code == 404:
            return None
        else:
            return None
    except requests.exceptions.RequestException:
        return None

import re

def parse_gwas_beta(beta_string):
    """
    Parses a GWAS Catalog beta string (e.g., "0.61413 unit decrease")
    Returns a tuple: (numeric_float, unit_description_string)
    """
    if not beta_string:
        return None, None

    beta_str = str(beta_string).strip()

    # Regex to find the first number (handles decimals and signs)
    match = re.search(r"[-+]?\d*\.?\d+", beta_str)

    if match:
        numeric_val = float(match.group())
        # Extract the rest of the string as the unit/direction
        description = beta_str.replace(match.group(), "").strip()

        # CRITICAL FIX: If the text says "decrease", the statistical beta is actually negative
        if "decrease" in description.lower() and numeric_val > 0:
            numeric_val = -numeric_val

        return numeric_val, description

    return None, beta_str # Fallback if no numbers are found

def enrich_graph_with_gwas(g, max_snps=5):
    print("\n--- Starting GWAS Catalog Integration (v2 API) ---")

    g.bind("efo", EFO)
    g.bind("biolink", BIOLINK)
    g.bind("prov", PROV)
    g.bind("sio", SIO)
    g.bind("pubmed", PUBMED)
    g.bind("skos", SKOS)

    snp_nodes = list(g.subjects(RDF.type, MTR.SNP))
    snps_to_process = snp_nodes[:max_snps]

    for i, snp_node in enumerate(snps_to_process):
        # .strip() prevents 404 errors caused by hidden whitespace
        rsid = str(snp_node).split('/')[-1].strip()

        print(f"[{i+1}/{len(snps_to_process)}] Fetching v2 associations for {rsid}...")

        endpoint = "v2/associations"
        params = {
            "rs_id": rsid,
            "sort": "p_value",
            "direction": "asc",
            "size": 10
        }

        associations_data = gwas_api_request(endpoint, params)

        # --- ROBUST ERROR HANDLING ---
        # Check if data exists AND if the nested keys we expect are actually there
        if not associations_data or '_embedded' not in associations_data or 'associations' not in associations_data['_embedded']:
            print("   -> No associated traits found in the GWAS Catalog for this SNP.")
            time.sleep(0.1)
            continue # Safely skip to the next SNP

        associations_list = associations_data['_embedded']['associations']

        # Double check that the list isn't empty
        if len(associations_list) == 0:
            print("   -> No associated traits found in the GWAS Catalog for this SNP.")
            time.sleep(0.1)
            continue

        print(f"   -> Found {len(associations_list)} significant associations!")

        for assoc in associations_list:
            # Extract core fields
            p_value = assoc.get('p_value')
            beta = assoc.get('beta')
            ci_lower = assoc.get('ci_lower')
            ci_upper = assoc.get('ci_upper')
            effect_allele = assoc.get('snp_effect_allele')

            # Extract provenance fields
            pubmed_id = assoc.get('pubmed_id')
            accession_id = assoc.get('accession_id')
            reported_trait = assoc.get('reported_trait')

            efo_traits = assoc.get('efo_traits', [])

            for trait_dict in efo_traits:
                trait_name = trait_dict.get('efo_trait')
                if not trait_name:
                    continue

                # Create Trait Node
                safe_trait_id = urllib.parse.quote(trait_name.strip().replace(" ", "_").replace('"', ''))
                trait_node = EFO[f"Custom_{safe_trait_id}"]

                # Add Trait Core Identity
                g.add((trait_node, RDF.type, BIOLINK.DiseaseOrPhenotypicFeature))
                g.add((trait_node, RDFS.label, Literal(trait_name)))

                # If author reported a different name, save it as an alternative label!
                if reported_trait and reported_trait != trait_name:
                    g.add((trait_node, SKOS.altLabel, Literal(reported_trait)))

                # Create the Association Node
                # We use the accession_id to ensure every unique GWAS study gets its own distinct edge
                unique_assoc_id = accession_id if accession_id else f"Assoc_{rsid}_{safe_trait_id}"
                assoc_node = MTR[f"GWAS_{unique_assoc_id}"]

                g.add((assoc_node, RDF.type, BIOLINK.Association))
                g.add((assoc_node, BIOLINK.has_subject, snp_node))
                g.add((assoc_node, BIOLINK.has_object, trait_node))

                # --- ADDING THE NEW STATISTICAL FIELDS ---
        # --- ADDING THE NEW STATISTICAL FIELDS ---
                if p_value is not None:
                    g.add((assoc_node, MTR.p_value, Literal(p_value, datatype=XSD.float)))

                # --- NEW BETA PARSING LOGIC ---
                if beta is not None:
                    numeric_beta, beta_unit = parse_gwas_beta(beta)

                    if numeric_beta is not None:
                        # Store the mathematically pure float
                        g.add((assoc_node, MTR.beta, Literal(numeric_beta, datatype=XSD.float)))

                    if beta_unit:
                        # Store the unit/context as a separate string so you don't lose the biological meaning!
                        g.add((assoc_node, MTR.beta_unit, Literal(beta_unit)))
                # ------------------------------

                if ci_lower is not None and ci_upper is not None:
                    g.add((assoc_node, MTR.ci_lower, Literal(ci_lower, datatype=XSD.float)))
                    g.add((assoc_node, MTR.ci_upper, Literal(ci_upper, datatype=XSD.float)))
                if effect_allele:
                    g.add((assoc_node, MTR.effect_allele, Literal(effect_allele)))

                # --- ADDING THE NEW PROVENANCE FIELDS ---
                g.add((assoc_node, PROV.wasGeneratedBy, Literal("EBI_GWAS_REST_API_v2")))

                if pubmed_id:
                    # Link the association to the exact PubMed article
                    paper_node = PUBMED[str(pubmed_id)]
                    g.add((assoc_node, BIOLINK.publications, paper_node))

                if accession_id:
                    # Link the association to the specific GWAS Catalog Study
                    study_node = URIRef(f"https://www.ebi.ac.uk/gwas/studies/{accession_id}")
                    g.add((assoc_node, PROV.wasDerivedFrom, study_node))

        time.sleep(0.1)

    print("\n--- GWAS Integration Complete ---")
    return g

In [113]:
g

<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>

In [114]:
associations_df.columns

Index(['association_id', 'risk_frequency', 'pvalue_description', 'range',
       'beta', 'p_value', 'efo_traits', 'reported_trait', 'accession_id',
       'locations', 'mapped_genes', 'bg_efo_traits', 'pubmed_id',
       'first_author', 'ci_lower', 'ci_upper', '_links', 'snp_effect_allele',
       'snp_allele', 'trait_name'],
      dtype='str')

In [115]:
associations_df.columns

Index(['association_id', 'risk_frequency', 'pvalue_description', 'range',
       'beta', 'p_value', 'efo_traits', 'reported_trait', 'accession_id',
       'locations', 'mapped_genes', 'bg_efo_traits', 'pubmed_id',
       'first_author', 'ci_lower', 'ci_upper', '_links', 'snp_effect_allele',
       'snp_allele', 'trait_name'],
      dtype='str')

In [116]:
print(f"Graph size after GWAS enrichment: {len(g)} triples.")


Graph size after GWAS enrichment: 233 triples.


In [117]:
# 3. Enrich with GWAS Catalog (Testing with 5 SNPs)
g = enrich_graph_with_gwas(g, max_snps=5)
print(f"Graph size after GWAS enrichment: {len(g)} triples.")


--- Starting GWAS Catalog Integration (v2 API) ---
[1/3] Fetching v2 associations for rs102275...
   -> Found 10 significant associations!
[2/3] Fetching v2 associations for rs12928099...
   -> Found 10 significant associations!
[3/3] Fetching v2 associations for rs7412...
   -> Found 10 significant associations!

--- GWAS Integration Complete ---
Graph size after GWAS enrichment: 634 triples.


In [118]:
def run_sample_pleiotropy_query(g):
    print("\n--- Running Sample SPARQL Query: Pleiotropic Effects ---")

    query = """
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX biolink: <https://w3id.org/biolink/vocab/>
    PREFIX oban: <http://purl.org/oban/>
    PREFIX mtr: <http://example.org/rqtl/>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    SELECT ?snp ?ratio_name ?trait_name ?gwas_p_value ?gwas_beta
    WHERE {
      ?rqtl_assoc a oban:association ;
                  oban:has_subject ?snp ;
                  oban:has_object ?ratio .
      ?ratio rdfs:label ?ratio_name .

      ?gwas_assoc a biolink:Association ;
                  biolink:has_subject ?snp ;
                  biolink:has_object ?trait ;
                  mtr:p_value ?gwas_p_value .
      ?trait rdfs:label ?trait_name .

      OPTIONAL { ?gwas_assoc mtr:beta ?gwas_beta . }

      FILTER (?gwas_p_value < 0.00000005)
    }
    ORDER BY ?gwas_p_value
    LIMIT 10
    """

    # Execute the query on your rdflib Graph
    results = g.query(query)

    print(f"{'SNP':<35} | {'METABOLITE RATIO':<35} | {'GWAS DISEASE TRAIT':<40} | {'P-VALUE':<12} | {'BETA'}")
    print("-" * 140)

    for row in results:
        snp_id = str(row.snp).split('/')[-1] # Clean up the URI for display
        ratio = str(row.ratio_name)
        trait = str(row.trait_name)
        pval = f"{float(row.gwas_p_value):.2e}"
        beta = f"{float(row.gwas_beta):.3f}" if row.gwas_beta else "N/A"

        print(f"{snp_id:<35} | {ratio:<35} | {trait:<40} | {pval:<12} | {beta}")

# To call it in your main block:
# run_sample_pleiotropy_query(g)

In [119]:
run_sample_pleiotropy_query(g)


--- Running Sample SPARQL Query: Pleiotropic Effects ---
SNP                                 | METABOLITE RATIO                    | GWAS DISEASE TRAIT                       | P-VALUE      | BETA
--------------------------------------------------------------------------------------------------------------------------------------------
rs102275                            | 1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to lignoceroyl sphingomyelin (d18:1/24:0) | 1-palmitoyl-2-arachidonoyl-GPC (16:0/20:4n6) measurement | 0.00e+00     | -0.614
rs102275                            | 1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to lignoceroyl sphingomyelin (d18:1/24:0) | cholesteryl esters to total lipids in very small VLDL percentage  | 0.00e+00     | 0.080
rs102275                            | 1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to lignoceroyl sphingomyelin (d18:1/24:0) | free cholesterol to total lipids in large HDL percentage  | 0.00e+00     | -0.115
rs102275                            | 1-palm

In [122]:
import requests
import time
import urllib.parse
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import RDFS, XSD

# --- Namespaces ---
CHEMBL = Namespace("http://identifiers.org/chembl.compound/")
HGNC = Namespace("http://identifiers.org/hgnc.symbol/")
BIOLINK = Namespace("https://w3id.org/biolink/vocab/")
PROV = Namespace("http://www.w3.org/ns/prov#")
EFO = Namespace("http://www.ebi.ac.uk/efo/") # Added for safety liability events
MTR = Namespace("http://example.org/rqtl/")

OPENTARGETS_API = "https://api.platform.opentargets.org/api/v4/graphql"

def fetch_opentargets_drugs(gene_symbol):
    """
    Executes a GraphQL query against Open Targets to find known drugs,
    tractability, and safety liabilities for a specific gene symbol.
    """
    query = """
    query searchDrugTargets($symbol: String!) {
      search(queryString: $symbol, entityNames: ["target"], page: {index: 0, size: 1}) {
        hits {
          id
          object {
            ... on Target {
              approvedSymbol

              # 1. Fetch Tractability (Druggability)
              tractability {
                smallmolecule {
                  value
                }
                antibody {
                  value
                }
              }

              # 2. Fetch Safety Liabilities (Adverse Events)
              safetyLiabilities {
                event
                biosample {
                  tissueLabel
                }
              }

              # 3. Fetch Known Drugs
              knownDrugs(size: 10) {
                rows {
                  phase
                  drug {
                    id
                    name
                  }
                  disease {
                    name
                  }
                }
              }
            }
          }
        }
      }
    }
    """

    variables = {"symbol": gene_symbol}

    try:
        response = requests.post(OPENTARGETS_API, json={"query": query, "variables": variables})
        if response.status_code == 200:
            return response.json()
        return None
    except Exception as e:
        print(f"Open Targets API Error: {e}")
        return None

def enrich_graph_with_opentargets(g, max_genes=5):
    """
    Scans the graph for causal Genes, queries Open Targets,
    and adds known drugs, clinical phases, tractability, and safety data to the graph.
    """
    print("\n--- Starting Open Targets Integration ---")

    g.bind("chembl", CHEMBL)
    g.bind("biolink", BIOLINK)
    g.bind("prov", PROV)
    g.bind("efo", EFO)

    # 1. Find all Genes currently in the graph
    gene_nodes = list(g.subjects(RDF.type, MTR.Gene))
    print(f"Found {len(gene_nodes)} causal Genes in the local graph.")

    genes_to_process = gene_nodes[:max_genes]

    for i, gene_node in enumerate(genes_to_process):
        gene_symbol = str(gene_node).split('/')[-1].strip()

        print(f"[{i+1}/{len(genes_to_process)}] Fetching OT data for {gene_symbol}...")

        data = fetch_opentargets_drugs(gene_symbol)

        # 2. Parse the deeply nested GraphQL JSON response
        if data and 'data' in data and data['data']['search']['hits']:
            target_obj = data['data']['search']['hits'][0]['object']

            # --- A. PARSE TRACTABILITY ---
            tractability = target_obj.get('tractability', {})
            if tractability:
                sm_tractable = tractability.get('smallmolecule', {}).get('value')
                ab_tractable = tractability.get('antibody', {}).get('value')

                # If there is a True value, add it as a boolean property
                if sm_tractable is True:
                    g.add((gene_node, MTR.small_molecule_tractable, Literal(True, datatype=XSD.boolean)))
                if ab_tractable is True:
                    g.add((gene_node, MTR.antibody_tractable, Literal(True, datatype=XSD.boolean)))

            # --- B. PARSE SAFETY LIABILITIES ---
            safety_liabilities = target_obj.get('safetyLiabilities')
            if safety_liabilities:
                print(f"   -> Found {len(safety_liabilities)} safety liabilities for {gene_symbol}!")
                for liability in safety_liabilities:
                    event_name = liability.get('event')
                    if not event_name:
                        continue

                    # Create an Event Node
                    safe_event_id = urllib.parse.quote(event_name.strip().replace(" ", "_").replace('"', ''))
                    event_node = EFO[f"Custom_Event_{safe_event_id}"]

                    g.add((event_node, RDF.type, BIOLINK.PhenotypicFeature))
                    g.add((event_node, RDFS.label, Literal(event_name)))

                    # Link the liability to the gene
                    g.add((gene_node, BIOLINK.has_adverse_event, event_node))

                    # Add tissue context if available
                    biosample = liability.get('biosample')
                    if biosample and biosample.get('tissueLabel'):
                        g.add((event_node, MTR.tissue_context, Literal(biosample['tissueLabel'])))

            # --- C. PARSE KNOWN DRUGS ---
            if 'knownDrugs' in target_obj and target_obj['knownDrugs'] is not None:
                drug_rows = target_obj['knownDrugs']['rows']
                if drug_rows:
                    print(f"   -> Found {len(drug_rows)} known drugs targeting {gene_symbol}!")

                    for row in drug_rows:
                        drug_info = row['drug']
                        chembl_id = drug_info['id']
                        drug_name = drug_info['name']
                        phase = row['phase']

                        drug_node = CHEMBL[chembl_id]
                        g.add((drug_node, RDF.type, BIOLINK.Drug))
                        g.add((drug_node, RDFS.label, Literal(drug_name)))
                        g.add((drug_node, MTR.max_clinical_phase, Literal(phase, datatype=XSD.float)))
                        g.add((drug_node, BIOLINK.targets, gene_node))
                        g.add((drug_node, PROV.wasDerivedFrom, URIRef("https://platform.opentargets.org/")))
                else:
                    print(f"   -> No drugs found currently targeting {gene_symbol}.")
            else:
                 print(f"   -> Gene {gene_symbol} is novel! No known drugs found.")

        else:
            print(f"   -> {gene_symbol} not found in Open Targets database.")

        time.sleep(0.1)

    print("\n--- Open Targets Integration Complete ---")
    return g

In [123]:
g = enrich_graph_with_opentargets(g, max_genes=5)


--- Starting Open Targets Integration ---
Found 13 causal Genes in the local graph.
[1/5] Fetching OT data for TMEM258...
   -> TMEM258 not found in Open Targets database.
[2/5] Fetching OT data for FADS1...
   -> FADS1 not found in Open Targets database.
[3/5] Fetching OT data for FADS2...
   -> FADS2 not found in Open Targets database.
[4/5] Fetching OT data for RP11-680G24.5...
   -> RP11-680G24.5 not found in Open Targets database.
[5/5] Fetching OT data for NPIP...
   -> NPIP not found in Open Targets database.

--- Open Targets Integration Complete ---


In [125]:
pip install pyvis

  Using cached pyvis-0.3.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jsonpickle-4.1.1-py3-none-any.whl.metadata (8.1 kB)
Using cached pyvis-0.3.2-py3-none-any.whl (756 kB)
Using cached jsonpickle-4.1.1-py3-none-any.whl (47 kB)

   ---------------------------------------- 0/4 [MarkupSafe]
   ---------------------------------------- 0/4 [MarkupSafe]
   ---------------------------------------- 0/4 [MarkupSafe]
   ---------- ----------------------------- 1/4 [jsonpickle]
   ---------- ----------------------------- 1/4 [jsonpickle]
   ---------- ----------------------------- 1/4 [jsonpickle]
   ---------- ----------------------------- 1/4 [jsonpickle]
   -------------------- ------------------- 2/4 [jinja2]
   -------------------- ------------------- 2/4 [jinja2]
   -------------------- ------------------- 2/4 [jinja2]
   -------------------- ------------------- 2/4 [jinja2]
   -------------------- ------------------- 2/4 [jinja2]
   -------------------- ------------------- 2/4 [jin


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [127]:
# pip install pyvis networkx
from pyvis.network import Network
import networkx as nx

def visualize_graph(rdf_graph, max_nodes=100):
    # 1. Create a PyVis network
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True)

    # 2. Extract triples from your rdflib graph
    count = 0
    for subj, pred, obj in rdf_graph:
        if count >= max_nodes:
            break

        # Clean up the URIs so they look readable on the screen
        s_label = str(subj).split('/')[-1].replace("Custom_", "")
        o_label = str(obj).split('/')[-1].replace("Custom_", "")
        p_label = str(pred).split('/')[-1]

        # Add nodes and edges to the interactive network
        net.add_node(s_label, label=s_label, title=str(subj))
        net.add_node(o_label, label=o_label, title=str(obj))
        net.add_edge(s_label, o_label, title=p_label, label=p_label)

        count += 1

    # 3. Generate the interactive HTML file
    net.show_buttons(filter_=['physics']) # Adds sliders to adjust gravity and spacing!
    net.show("my_knowledge_graph.html")
    print("Interactive graph saved to my_knowledge_graph.html")

# Run it on your graph!
visualize_graph(g, max_nodes=150)

my_knowledge_graph.html
Interactive graph saved to my_knowledge_graph.html


In [128]:
import requests
import networkx as nx
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from rdflib.namespace import RDFS

# --- New Namespaces ---
REACTOME = Namespace("https://reactome.org/content/detail/")
BIOLINK = Namespace("https://w3id.org/biolink/vocab/")
CHEBI = Namespace("http://purl.obolibrary.org/obo/CHEBI_")
MTR = Namespace("http://example.org/rqtl/")

def fetch_reactome_reactions(chebi_id):
    """
    Queries the Reactome Content Service for reactions involving a specific ChEBI ID.
    """
    # Reactome's mapping endpoint for external identifiers
    url = f"https://reactome.org/ContentService/data/mapping/ChEBI/{chebi_id}"

    try:
        response = requests.get(url, headers={"Accept": "application/json"})
        if response.status_code == 200:
            data = response.json()
            reactions = []

            # The API returns a list of events (pathways and reactions)
            for event in data:
                # Filter strictly for Reactions (not broad pathways)
                if event.get('className') == 'Reaction' and event.get('speciesName') == 'Homo sapiens':
                    reactions.append({
                        "id": event['stId'], # e.g., R-HSA-123456
                        "name": event['displayName']
                    })
            return reactions
        return []
    except Exception as e:
        print(f"Reactome API Error for {chebi_id}: {e}")
        return []

def enrich_graph_with_reactome(g):
    """
    Scans the graph for Metabolites, fetches their Reactome reactions,
    and links them in the Knowledge Graph.
    """
    print("\n--- Starting Reactome Integration ---")
    g.bind("reactome", REACTOME)

    # 1. Find all Metabolites in the graph
    metabolite_nodes = list(g.subjects(RDF.type, MTR.Metabolite))
    print(f"Found {len(metabolite_nodes)} Metabolites. Querying Reactome...")

    for node in metabolite_nodes:
        # Extract the numeric ID (e.g., "http://.../CHEBI_27747" -> "27747")
        chebi_id = str(node).split('_')[-1]

        reactions = fetch_reactome_reactions(chebi_id)
        if reactions:
            print(f"   -> {chebi_id}: Found {len(reactions)} reactions.")
            for rxn in reactions:
                rxn_node = REACTOME[rxn['id']]

                # Add Reaction Node
                g.add((rxn_node, RDF.type, BIOLINK.MolecularActivity))
                g.add((rxn_node, RDFS.label, Literal(rxn['name'])))

                # Link Metabolite -> Reaction
                g.add((node, BIOLINK.participates_in, rxn_node))

    print("--- Reactome Integration Complete ---\n")
    return g

In [129]:
enrich_graph_with_reactome(g)


--- Starting Reactome Integration ---
Found 2 Metabolites. Querying Reactome...
--- Reactome Integration Complete ---



<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>

In [130]:
def calculate_reaction_distances(g):
    """
    Converts the RDF graph to a NetworkX graph to calculate the
    shortest reaction path between the numerator and denominator of each ratio.
    """
    print("\n--- Calculating Metabolic Reaction Distances ---")

    # 1. Convert rdflib Graph to an undirected NetworkX graph
    # We only care about connections, not directionality, for distance
    nx_graph = nx.Graph()
    for subj, pred, obj in g:
        # We only add nodes that are URIs (ignore literal strings/numbers)
        if isinstance(subj, URIRef) and isinstance(obj, URIRef):
            nx_graph.add_edge(subj, obj, predicate=pred)

    # 2. Find all Metabolite Ratios
    ratio_nodes = list(g.subjects(RDF.type, MTR.MetaboliteRatio))

    for ratio in ratio_nodes:
        ratio_name = g.value(subject=ratio, predicate=RDFS.label)

        # Get Numerator and Denominator
        numerator = g.value(subject=ratio, predicate=MTR.has_numerator)
        denominator = g.value(subject=ratio, predicate=MTR.has_denominator)

        if numerator and denominator:
            num_name = g.value(subject=numerator, predicate=RDFS.label)
            den_name = g.value(subject=denominator, predicate=RDFS.label)

            try:
                # 3. Calculate the shortest path using NetworkX
                # NetworkX will traverse: Metabolite_A -> Reaction -> Metabolite_B
                path = nx.shortest_path(nx_graph, source=numerator, target=denominator)

                # The "Reaction Distance" is the number of steps.
                # If path is [Metab_A, Rxn_1, Metab_B], distance is 1 reaction.
                # Distance = (len(path) - 1) / 2
                reaction_distance = (len(path) - 1) // 2

                print(f"Ratio: {ratio_name}")
                print(f"  Path: {num_name} -> {den_name}")
                print(f"  Reaction Distance: {reaction_distance} steps")

                # 4. Save this newly discovered insight BACK into the RDF Graph!
                g.add((ratio, MTR.reaction_distance, Literal(reaction_distance, datatype=XSD.integer)))

            except nx.NetworkXNoPath:
                print(f"Ratio: {ratio_name}")
                print(f"  Reaction Distance: INF (No known pathway connecting them in Reactome)")

    return g

In [131]:
calculate_reaction_distances(g)


--- Calculating Metabolic Reaction Distances ---
Ratio: 1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) to lignoceroyl sphingomyelin (d18:1/24:0)
  Path: 1-palmitoyl-2-linoleoyl-gpc (16:0/18:2) -> lignoceroyl sphingomyelin (d18:1/24:0)
  Reaction Distance: 1 steps


<Graph identifier=Nd8ad1b36d1934a51ba23f7c6aae9ca3f (<class 'rdflib.graph.Graph'>)>